# Introduction to DSPy

DSPy is a declarative way to build with LLMs. It helps us program LLMs, rather than prompting them, creating modular, maintainable and optimizable AI software.

## What we’ll learn today

In this tutorial we’ll build a haiku-writing program (kind of English poetry) that starts with four lines of Python and grows into a tool-using, prompt-optimized agent. Along the way we’ll touch each of DSPy’s core components. We’ll learn:

- How to install DSPy, configure a **language model** and write a simple DSPy program.
- What a **Signature** is, and why DSPy uses signatures instead of hand-written prompt strings.
- What a **Module** is, how `Predict`, `ChainOfThought`, and `ReAct` differ, and when to reach for each.
- How to compose a custom `dspy.Module` that decomposes a task into named, independent stages.
- How to write **metrics** and use **optimizers** to compile better versions of our program.
- How to save optimized programs and reload them.

> A **haiku** is a traditional form of Japanese poetry characterized by its brevity and structural constraints. It captures a fleeting moment in time, traditionally focusing on nature or the seasons.

Example:

```
An old silent pond
A frog jumps into the pond,
splash! Silence again.
— Matsuo Bashō (translated)
```

# Setting up DSPy

## Load environment variables

In [13]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


## Install Library

In [ ]:
# ! uv add dspy
# ! pip install dspy


## Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [14]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

Behind the scenes, DSPy uses the [LiteLLM](https://docs.litellm.ai/docs/#litellm-python-sdk) library to normalize inference providers into a single format. This allows you to provide a LiteLLM model string and connect to nearly any model and its provider. [Click here to search for the full list](https://models.litellm.ai/).

For example, to connect directly use the string format:

```py
lm = dspy.LM("openai/gpt-5-mini")
lm = dspy.LM("anthropic/claude-sonnet-4-6")
```

We'll use [**OpenRouter**](https://openrouter.ai/); a unified interface for LLMs.

Key benefits include:

- Circumvent regional restrictions
- Some models are free
- One API for all models
    - switch easily between models and providers by changing a `str` value
    - no subscription to each provider needed

![Open Router ](../assets/open_router.png){height=256}

### Test run

Once we have an `LM`, calling `dspy.configure(lm=lm)` sets our `LM` as the default provider for every DSPy program in the process. This sets our `LM` globally, but we can selectively override this with [`dspy.context`](../diving-deeper/settings-and-context.md) when more granular control is needed.


Let’s ensure everything works by manually calling the model:

In [15]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "What is the capital of Saudi Arabia?"
    }
]

In [16]:
response = lm(messages=messages)
print(response)

['The capital of Saudi Arabia is Riyadh.']


With an `LM` configured, we can proceed to writing our first program.